In [0]:
spark.sql("USE CATALOG coretec_dev")

df_acessos = spark.table("coretec_dev.silver.acessos_portaria")
df_ocorrencias = spark.table("coretec_dev.silver.ocorrencias")
df_condominios = spark.table("coretec_dev.silver.condominios")

In [0]:
tabelas = {
    "acessos_portaria": df_acessos,
    "ocorrencias": df_ocorrencias,
    "condominios": df_condominios
}

for nome, df in tabelas.items():
    print(f"\n===== {nome} =====")
    print(f"Registros: {df.count()}")
    print(f"Colunas: {len(df.columns)}")
    df.printSchema()

In [0]:
display(df_acessos.limit(5))
display(df_ocorrencias.limit(5))
display(df_condominios.limit(5))

In [0]:
df_dim_condominio = df_condominios.select(
    "id_condominio",
    "nome_condominio",
    "bairro",
    "cidade",
    "uf",
    "tipo_condominio",
    "qtd_unidades",
    "plano_contratado",
    "status_contrato",
    "data_inicio_contrato",
    "valor_mensalidade_base"
)

display(df_dim_condominio.limit(10))

In [0]:
df_dim_condominio.printSchema()

In [0]:
(
    df_dim_condominio.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("coretec_dev.gold.dim_condominio")
)

In [0]:
df_dim_condominio_gold = spark.table("coretec_dev.gold.dim_condominio")

print("Registros:", df_dim_condominio_gold.count())
display(df_dim_condominio_gold.limit(10))


In [0]:
spark.sql("""
DESCRIBE DETAIL coretec_dev.gold.dim_condominio
""").display()

In [0]:
spark.sql("""
DESCRIBE HISTORY coretec_dev.gold.dim_condominio
""").display()

In [0]:
print("Acessos:")
print(df_acessos.columns)

print("\nOcorrências:")
print(df_ocorrencias.columns)

In [0]:
from pyspark.sql.functions import to_date, min, max

df_datas = (
    df_acessos
    .select(to_date("data_hora_acesso").alias("data"))
    .unionByName(
        df_ocorrencias.select(to_date("data_ocorrencia").alias("data"))
    )
)

df_datas.agg(
    min("data").alias("menor_data"),
    max("data").alias("maior_data")
).display()

In [0]:
from pyspark.sql.functions import sequence, explode, lit

df_dim_data = (
    spark.range(1)
    .select(
        explode(
            sequence(
                lit("2026-01-01").cast("date"),
                lit("2026-05-21").cast("date")
            )
        ).alias("data")
    )
)

display(df_dim_data)

In [0]:
from pyspark.sql.functions import (
    year, month, quarter, date_format,
    dayofweek, when
)

df_dim_data = (
    df_dim_data
    .withColumn("chave_data", date_format("data", "yyyyMMdd").cast("int"))
    .withColumn("ano", year("data"))
    .withColumn("mes", month("data"))
    .withColumn("nome_mes", date_format("data", "MMMM"))
    .withColumn("trimestre", quarter("data"))
    .withColumn("ano_mes", date_format("data", "yyyy-MM"))
    .withColumn("dia_semana", date_format("data", "EEEE"))
    .withColumn(
        "fim_de_semana",
        when(dayofweek("data").isin(1, 7), True).otherwise(False)
    )
    .select(
        "chave_data",
        "data",
        "ano",
        "mes",
        "nome_mes",
        "trimestre",
        "ano_mes",
        "dia_semana",
        "fim_de_semana"
    )
)

display(df_dim_data.limit(10))

In [0]:
print("Registros:", df_dim_data.count())
df_dim_data.printSchema()

In [0]:
(
    df_dim_data.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("coretec_dev.gold.dim_data")
)

In [0]:
df_dim_data_gold = spark.table("coretec_dev.gold.dim_data")

print("Registros gravados:", df_dim_data_gold.count())
display(df_dim_data_gold.limit(5))

In [0]:
spark.sql("""
DESCRIBE DETAIL coretec_dev.gold.dim_data
""").display()

In [0]:
spark.sql("""
DESCRIBE HISTORY coretec_dev.gold.dim_data
""").display()

In [0]:
df_acessos.printSchema()

In [0]:
from pyspark.sql.functions import date_format, lit

df_fato_acessos = (
    df_acessos
    .select(
        "id_acesso",
        "id_condominio",
        "id_operador",
        "data_hora_acesso",
        "tipo_pessoa",
        "tipo_acesso",
        "status_acesso",
        "canal_autorizacao",
        "tempo_atendimento_segundos"
    )
    .withColumn(
        "chave_data",
        date_format("data_hora_acesso", "yyyyMMdd").cast("int")
    )
    .withColumn("quantidade_acesso", lit(1))
)

display(df_fato_acessos.limit(10))

In [0]:
from pyspark.sql.functions import date_format, lit

df_fato_acessos = df_acessos.select(
    "id_acesso",
    "id_condominio",
    date_format("data_hora_acesso", "yyyyMMdd").cast("int").alias("chave_data"),
    "id_operador",
    "data_hora_acesso",
    "tipo_pessoa",
    "tipo_acesso",
    "status_acesso",
    "canal_autorizacao",
    "tempo_atendimento_segundos",
    lit(1).alias("quantidade_acesso")
)

display(df_fato_acessos.limit(10))

In [0]:
df_dim_condominio_gold = spark.table("coretec_dev.gold.dim_condominio")
df_dim_data_gold = spark.table("coretec_dev.gold.dim_data")

print("Silver:", df_acessos.count())
print("Fato preparada:", df_fato_acessos.count())

print(
    "IDs de acesso duplicados:",
    df_fato_acessos
    .groupBy("id_acesso")
    .count()
    .filter("count > 1")
    .count()
)

print(
    "Chaves principais ou estrangeiras nulas:",
    df_fato_acessos
    .filter("""
        id_acesso IS NULL
        OR id_condominio IS NULL
        OR chave_data IS NULL
    """)
    .count()
)

print(
    "Condomínios sem correspondência:",
    df_fato_acessos
    .select("id_condominio")
    .distinct()
    .join(
        df_dim_condominio_gold.select("id_condominio"),
        "id_condominio",
        "left_anti"
    )
    .count()
)

print(
    "Datas sem correspondência:",
    df_fato_acessos
    .select("chave_data")
    .distinct()
    .join(
        df_dim_data_gold.select("chave_data"),
        "chave_data",
        "left_anti"
    )
    .count()
)

df_fato_acessos.printSchema()

In [0]:
(
    df_fato_acessos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("coretec_dev.gold.fato_acessos_portaria")
)

df_fato_acessos_gold = spark.table(
    "coretec_dev.gold.fato_acessos_portaria"
)

print("Registros gravados:", df_fato_acessos_gold.count())
display(df_fato_acessos_gold.limit(10))

display(
    spark.sql("""
        DESCRIBE DETAIL coretec_dev.gold.fato_acessos_portaria
    """)
)

display(
    spark.sql("""
        DESCRIBE HISTORY coretec_dev.gold.fato_acessos_portaria
    """)
)

In [0]:
df_fato_ocorrencias = df_ocorrencias.select(
    "id_ocorrencia",
    "id_condominio",
    date_format("data_ocorrencia", "yyyyMMdd").cast("int").alias("chave_data"),
    "id_operador",
    "data_ocorrencia",
    "tipo_ocorrencia",
    "prioridade",
    "status_ocorrencia",
    "tempo_resolucao_minutos",
    "descricao_resumida",
    lit(1).alias("quantidade_ocorrencia")
)

display(df_fato_ocorrencias.limit(10))

In [0]:
print("Silver:", df_ocorrencias.count())
print("Fato preparada:", df_fato_ocorrencias.count())

print(
    "IDs de ocorrência duplicados:",
    df_fato_ocorrencias
    .groupBy("id_ocorrencia")
    .count()
    .filter("count > 1")
    .count()
)

print(
    "Chaves principais ou estrangeiras nulas:",
    df_fato_ocorrencias
    .filter("""
        id_ocorrencia IS NULL
        OR id_condominio IS NULL
        OR chave_data IS NULL
    """)
    .count()
)

print(
    "Condomínios sem correspondência:",
    df_fato_ocorrencias
    .select("id_condominio")
    .distinct()
    .join(
        df_dim_condominio_gold.select("id_condominio"),
        "id_condominio",
        "left_anti"
    )
    .count()
)

print(
    "Datas sem correspondência:",
    df_fato_ocorrencias
    .select("chave_data")
    .distinct()
    .join(
        df_dim_data_gold.select("chave_data"),
        "chave_data",
        "left_anti"
    )
    .count()
)

df_fato_ocorrencias.printSchema()

In [0]:
(
    df_fato_ocorrencias.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("coretec_dev.gold.fato_ocorrencias")
)

df_fato_ocorrencias_gold = spark.table(
    "coretec_dev.gold.fato_ocorrencias"
)

print("Registros gravados:", df_fato_ocorrencias_gold.count())
display(df_fato_ocorrencias_gold.limit(10))

display(
    spark.sql("""
        DESCRIBE DETAIL coretec_dev.gold.fato_ocorrencias
    """)
)

display(
    spark.sql("""
        DESCRIBE HISTORY coretec_dev.gold.fato_ocorrencias
    """)
)

In [0]:
tabelas_gold = [
    "coretec_dev.gold.dim_condominio",
    "coretec_dev.gold.dim_data",
    "coretec_dev.gold.fato_acessos_portaria",
    "coretec_dev.gold.fato_ocorrencias"
]

for tabela in tabelas_gold:
    quantidade = spark.table(tabela).count()
    print(f"{tabela}: {quantidade} registros")